In [4]:
import mlbstatsapi

mlb = mlbstatsapi.Mlb()

player_name = "Aaron Judge"
ids = mlb.get_people_id(player_name)   # returns a list of matching ids
if not ids:
    raise ValueError(f"No player found for name={player_name!r}")

person_id = ids[0]
print(f"Person ID for {player_name!r} is {person_id}")


Person ID for 'Aaron Judge' is 592450


In [10]:
splits = mlb.get_player_stats(
    person_id,
    stats=["season"],
    groups=["hitting"],
    season="2024",
)

print(f"Splits for {player_name!r} in 2024:")
for thing in splits["hitting"]["season"]:
    print(type(thing))
    print(thing)

Splits for 'Aaron Judge' in 2024:
<class 'tuple'>
('group', 'hitting')
<class 'tuple'>
('type', 'season')
<class 'tuple'>
('total_splits', 1)
<class 'tuple'>
('exemptions', [])
<class 'tuple'>
('splits', [HittingSeason(season='2024', num_teams=None, num_leagues=None, game_type='R', rank=None, position=None, team=Team(id=147, link='/api/v1/teams/147', name='New York Yankees', spring_league=None, all_star_status=None, season=None, venue=None, spring_venue=None, team_code=None, file_code=None, abbreviation=None, team_name=None, location_name=None, first_year_of_play=None, league=None, division=None, sport=None, short_name=None, record=None, franchise_name=None, club_name=None, active=None, parent_org_name=None, parent_org_id=None), player=Person(id=592450, link='/api/v1/people/592450', primary_position=None, pitch_hand=None, bat_side=None, full_name='Aaron Judge', first_name=None, last_name=None, primary_number=None, birth_date=None, current_team=None, current_age=None, birth_city=None, b

## Fetch functions

`fetch_player_hitting_stats` and `fetch_player_pitching_stats` accept an MLBAM player ID and a season year, and return a dict keyed by the app's stat names.

**Note on K%:** `stat.strikeouts` is always `None` due to a camelCase mismatch in the mlbstatsapi model (`strikeouts` vs the API's `strikeOuts`). We work around this by pulling `strikeOuts` from the raw JSON response.

In [ ]:
def fetch_player_hitting_stats(player_id: int, season: int) -> dict | None:
    stats = mlb.get_player_stats(player_id, stats=["season"], groups=["hitting"], season=season)
    if not stats or "hitting" not in stats:
        return None

    splits = stats["hitting"]["season"].splits
    if not splits:
        return None
    stat = splits[0].stat

    # stat.strikeouts is always None (mlbstatsapi camelCase bug) — fetch raw for strikeOuts
    raw = mlb._mlb_adapter_v1.get(
        endpoint=f"people/{player_id}/stats",
        ep_params={"stats": ["season"], "group": ["hitting"], "season": season},
    ).data["stats"][0]["splits"][0]["stat"]

    pa = stat.plate_appearances
    return {
        "BA":  float(stat.avg),
        "OBP": float(stat.obp),
        "SLG": float(stat.slg),
        "OPS": float(stat.ops),
        "K%":  raw["strikeOuts"] / pa,
        "BB%": stat.base_on_balls / pa,
    }

In [2]:
def fetch_player_pitching_stats(player_id: int, season: int) -> dict | None:
    stats = mlb.get_player_stats(player_id, stats=["season"], groups=["pitching"], season=season)
    if not stats or "pitching" not in stats:
        return None

    splits = stats["pitching"]["season"].splits
    if not splits:
        return None
    stat = splits[0].stat

    return {
        "ERA":  stat.era,
        "WHIP": stat.whip,
        "SO9":  stat.strikeouts_per_9_inn,
        "SO/W": stat.strikeout_walk_ratio,
        "IP":   stat.innings_pitched,
    }

In [5]:
# Test: Aaron Judge hitting 2024 (ID from cell above)
judge_id = mlb.get_people_id("Aaron Judge")[0]
judge_hitting = fetch_player_hitting_stats(judge_id, 2024)
print("Aaron Judge 2024 hitting:")
for stat, val in judge_hitting.items():
    print(f"  {stat:<5} {val:.4f}")

print()

# Test: Zack Wheeler pitching 2025
wheeler_id = mlb.get_people_id("Zack Wheeler")[0]
wheeler_pitching = fetch_player_pitching_stats(wheeler_id, 2025)
print("Zack Wheeler 2025 pitching:")
for stat, val in wheeler_pitching.items():
    print(f"  {stat:<5} {val}")

age=32 games_played=158 flyouts=None groundouts=None airouts=None runs=122 doubles=36 triples=1 home_runs=58 strikeouts=None base_on_balls=133 intentional_walks=20 hits=180 hit_by_pitch=9 avg='.322' at_bats=559 obp='.458' slg='.701' ops='1.159' caught_stealing=0 caught_stealing_percentage='.000' stolen_bases=10 stolen_base_percentage='1.000' ground_into_double_play=22 ground_into_triple_play=None number_of_pitches=2885 plate_appearances=704 total_bases=392 rbi=144 left_on_base=238 sac_bunts=0 sac_flies=2 babip='.367' groundouts_to_airouts=None catchers_interference=1 at_bats_per_home_run='9.64'
Aaron Judge 2024 hitting:
  BA    0.3220
  OBP   0.4580
  SLG   0.7010
  OPS   1.1590
  K%    0.2429
  BB%   0.1889

Zack Wheeler 2025 pitching:
  ERA   2.71
  WHIP  0.94
  SO9   11.73
  SO/W  5.91
  IP    149.2
